# Bias–Variance Trade-off

> A practical, hands-on Jupyter notebook to understand and manage the bias–variance trade-off in machine learning.

**Badges**: ![Python](https://img.shields.io/badge/Python-3.10+-blue.svg) ![scikit-learn](https://img.shields.io/badge/scikit--learn-%3E=1.2-orange.svg) ![PyTorch](https://img.shields.io/badge/PyTorch-%3E=2.0-red.svg)


## Overview
The **bias–variance trade-off** describes the balance between two error sources in predictive modeling:

- **Bias**: Error from **overly simple** assumptions → underfitting (poor train & test performance).
- **Variance**: Error from **overly sensitive** models → overfitting (great train, poor test).

Goal: Find a model that’s **complex enough** to capture meaningful patterns yet **robust** enough to generalize.

## Quick Intuition

```text
Error
^
| \         test error
|  \      /\
|   \    /  \    <- variance dominates with high complexity
|    \  /    \
|     \ /      \_
|      \        \___
| bias  \___________\___> Model complexity
```


## Key Signs & Diagnostics
- **Underfitting (High Bias)**: Low train score, low test score; learning curves plateau early.
- **Overfitting (High Variance)**: High train score, low test score; large train–test gap.

**Recommended checks**: Train vs. validation metrics, learning curves, cross-validation, regularization paths, and model complexity sweeps.

## How to Reduce High Variance
1. **Simplify** the model (lower polynomial degree, reduce tree depth, fewer layers/units).
2. **Regularize** (L2/Ridge, L1/Lasso, Elastic Net).
3. **Dropout** (for neural nets).
4. **Early stopping**.
5. **Data augmentation** (images/text/audio).
6. **Cross-validation** to tune hyperparameters.
7. **Bagging/Ensembling** (Random Forests).

## How to Reduce High Bias
1. **Increase capacity** (higher polynomial degree, deeper trees, deeper/wider networks).
2. **Feature engineering** or richer architectures (e.g., CNNs for images).
3. **Tune hyperparameters** (learning rate, batch size, optimizers).
4. **Train longer** (monitor to avoid overfitting).
5. **Better optimizers** (Adam, RMSprop, momentum).

---
# Experiments (scikit-learn)

Below we generate a non-linear dataset and compare models with different complexity and regularization.

In [ ]:
# Setup: numpy, scikit-learn, matplotlib
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, KFold, cross_val_score

rng = np.random.RandomState(42)
X = np.sort(2 * np.pi * rng.rand(200, 1), axis=0)
y_true = np.sin(X).ravel()
noise = 0.3 * rng.randn(X.shape[0])
y = y_true + noise

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print('Train/Test sizes: {}/{}'
      .format(X_train.shape[0], X_test.shape[0]))


### Underfitting vs. Overfitting with Polynomial Regression
We sweep polynomial degrees and compare train/test MSE to observe bias and variance behavior.

In [ ]:
def fit_poly(degree):
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('lr', LinearRegression())
    ])
    model.fit(X_train, y_train)
    tr_mse = mean_squared_error(y_train, model.predict(X_train))
    te_mse = mean_squared_error(y_test,  model.predict(X_test))
    return tr_mse, te_mse

results = []
for d in [1, 3, 5, 9, 15]:
    tr, te = fit_poly(d)
    results.append((d, tr, te))
    print('Degree={:<2} | Train MSE={:.3f} | Test MSE={:.3f}'
      .format(d, tr, te))

# Plot train vs test error by degree
degrees = [r[0] for r in results]
train_mse = [r[1] for r in results]
test_mse  = [r[2] for r in results]
plt.figure(figsize=(6,4))
plt.plot(degrees, train_mse, '-o', label='Train MSE')
plt.plot(degrees, test_mse, '-o', label='Test MSE')
plt.xlabel('Polynomial Degree')
plt.ylabel('MSE')
plt.title('Bias–Variance via Polynomial Complexity')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


### Regularization: Ridge vs. Lasso
Increasing the regularization strength (alpha) usually **reduces variance** but may **increase bias**. Choose `alpha` via cross-validation.

In [ ]:
alphas = [1e-4, 1e-3, 1e-2, 1e-1, 1, 10]
ridge_scores = []
lasso_scores = []
for alpha in alphas:
    ridge = Pipeline([('sc', StandardScaler()), ('rg', Ridge(alpha=alpha))])
    lasso = Pipeline([('sc', StandardScaler()), ('ls', Lasso(alpha=alpha, max_iter=10000))])
    ridge.fit(X_train, y_train)
    lasso.fit(X_train, y_train)
    rmse_ridge_test = mean_squared_error(y_test, ridge.predict(X_test), squared=False)
    rmse_lasso_test = mean_squared_error(y_test, lasso.predict(X_test), squared=False)
    ridge_scores.append(rmse_ridge_test)
    lasso_scores.append(rmse_lasso_test)
    print('alpha={:<6} | Ridge RMSE={:.3f} | Lasso RMSE={:.3f}'
      .format(alpha, rmse_ridge_test, rmse_lasso_test))

plt.figure(figsize=(6,4))
plt.plot(alphas, ridge_scores, '-o', label='Ridge RMSE')
plt.plot(alphas, lasso_scores, '-o', label='Lasso RMSE')
plt.xscale('log')
plt.xlabel('alpha (log scale)')
plt.ylabel('RMSE on Test')
plt.title('Regularization Strength vs. Generalization')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


### Cross-Validation to Tune Complexity
Use cross-validation to select polynomial degree (or other complexity controls) by minimizing validation error.

In [ ]:
def poly_model(degree):
    return Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('lr', LinearRegression())
    ])

cv = KFold(n_splits=5, shuffle=True, random_state=42)
for d in range(1, 10):
    scores = cross_val_score(poly_model(d), X, y, cv=cv, scoring='neg_mean_squared_error')
    print('Degree {}: CV MSE = {:.3f} ± {:.3f}'
      .format(d, -scores.mean(), scores.std()))


---
# Experiments (PyTorch)

Demonstrations of **dropout** and **capacity increases** to manage variance and bias, respectively.

In [ ]:
# PyTorch setup
import torch
import torch.nn as nn
import torch.optim as optim

# Convert training data to tensors
X_tr = torch.tensor(X_train, dtype=torch.float32)
y_tr = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
X_te = torch.tensor(X_test, dtype=torch.float32)
y_te = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)

class MLP(nn.Module):
    def __init__(self, in_dim=1, hidden=128, p_dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(p_dropout),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(p_dropout),
            nn.Linear(hidden, 1)
        )
    def forward(self, x):
        return self.net(x)

model = MLP(in_dim=1, hidden=128, p_dropout=0.5)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)  # L2 regularization via weight_decay

def evaluate(model):
    model.eval()
    with torch.no_grad():
        te_preds = model(X_te)
        te_loss = criterion(te_preds, y_te).item()
    return te_loss

best_val = float('inf')
patience, wait = 20, 0
for epoch in range(300):
    model.train()
    optimizer.zero_grad()
    preds = model(X_tr)
    loss = criterion(preds, y_tr)
    loss.backward()
    optimizer.step()
    val = evaluate(model)
    if val < best_val - 1e-5:
        best_val = val
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print('Early stopping at epoch {} | Best Val MSE: {:.4f}'.format(epoch, best_val))
            break
print('Final Val MSE: {:.4f}'.format(evaluate(model)))


### Increasing Capacity to Reduce Bias
Adding layers/units can lower bias (capture complex patterns). Combine with regularization/early stopping to control variance.

In [ ]:
class BiggerMLP(nn.Module):
    def __init__(self, in_dim=1, hidden=256, depth=4):
        super().__init__()
        layers = [nn.Linear(in_dim, hidden), nn.ReLU()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden, hidden), nn.ReLU()]
        layers += [nn.Linear(hidden, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

big_model = BiggerMLP(in_dim=1, hidden=256, depth=4)
big_opt = optim.Adam(big_model.parameters(), lr=5e-4)
print(big_model)


---
## Practical Workflow Checklist
1. Split data into train/validation/test.
2. Start with a simple baseline.
3. Plot learning curves; compare train vs. validation performance.
4. Sweep complexity and regularization.
5. Use cross-validation for hyperparameter tuning.
6. Add regularization & dropout if variance is high.
7. Increase capacity if bias is high.
8. Apply early stopping; monitor validation metrics.
9. Document decisions & results.

## Notes & References
- *Elements of Statistical Learning* (Hastie, Tibshirani, Friedman) — classic text on bias–variance.
- scikit-learn User Guide — model selection, validation, regularization.
- Srivastava et al., 2014 — Dropout: A simple way to prevent neural networks from overfitting.

> Tip: Add direct links in your project readme to your preferred references (docs, papers, blog posts).